# Lab 3 — Reflexion: เรียนรู้จากความผิดพลาดโดยไม่ต้อง train (§19.2.3)

Agent ที่ดีต้องหยุดประเมินตัวเองได้ ตำราแบ่ง reflection เป็น 3 ระดับ:
1. **Output validation** — "ถูกไหม ลืมอะไรไหม"
2. **Trajectory review** — ย้อนดู *k* ขั้นล่าสุด
3. **Strategy revision** — "แก้ปัญหาถูกข้อไหม"

**Reflexion** เก็บบทเรียนจากความผิดพลาดไว้ใน *reflection memory* แล้วใส่ใน prompt รอบถัดไป
→ เรียนรู้ข้ามรอบได้ **โดยไม่อัปเดต weight**

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.2.3 — CC BY-SA 4.0

## 0. เตรียมเครื่อง

ต้องมี Python 3.10+ (หรือ Colab) และ **OpenRouter API key** — วิธีตั้งค่าอยู่ใน Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- จาก Lab 1: ตั้งค่า client + ask_llm ----
import os

from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"effort": "medium"}}

def ask_llm(prompt):
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA,
                                          messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content

## 1. งาน + Evaluator

Agent ต้องเขียนฟังก์ชัน `parse_duration()` แล้ว **evaluator** ตรวจด้วย unit tests ที่ agent **มองไม่เห็นล่วงหน้า**
(เหมือนงานจริงที่ requirement บางข้อไม่ได้เขียนไว้ในโจทย์)

In [ ]:
SPEC = ("Write a Python function `parse_duration(s: str) -> float` that converts a duration "
        "string like '1h30m' into minutes (e.g. '1h30m' -> 90). Return only the code in one "
        "```python block, no explanation.")

In [ ]:
HIDDEN_TESTS = [
    ("1h30m", 90), ("2h", 120), ("45m", 45),
    ("90s", 1.5),         # seconds
    ("1H 15M", 75),       # uppercase + spaces
    ("1.5h", 90),         # decimals
    ("1d", 1440),         # days
    ("abc", ValueError),  # invalid -> must raise ValueError
    ("10", ValueError),   # no unit -> ambiguous -> ValueError
]

In [ ]:
def extract_code(text):
    return text.split("```python")[-1].split("```")[0] if "```" in text else text

In [ ]:
def evaluate(code_str):
    ns = {}
    try:
        exec(code_str, ns)
        fn = ns["parse_duration"]
    except Exception as e:
        return [f"code does not run: {e}"]
    failures = []
    for inp, want in HIDDEN_TESTS:
        expects_error = isinstance(want, type) and issubclass(want, Exception)
        try:
            got = fn(inp)
            if expects_error:
                failures.append(f"parse_duration({inp!r}) returned {got}, expected {want.__name__}")
            elif abs(got - want) > 1e-9:
                failures.append(f"parse_duration({inp!r}) returned {got}, expected {want}")
        except Exception as e:
            if not (expects_error and isinstance(e, want)):
                failures.append(f"parse_duration({inp!r}) raised {type(e).__name__}: {e}")
    return failures

## 2. Reflexion loop

`attempt → evaluate → reflect → (บันทึกใน memory) → attempt ใหม่โดยอ่าน memory`

In [ ]:
def write_code(reflections):
    prompt = SPEC
    if reflections:
        prompt += "\n\nLessons from your previous failed attempts:\n"
        prompt += "\n".join(f"- {r}" for r in reflections)
    return extract_code(ask_llm(prompt))

In [ ]:
def reflect(code_str, failures):
    return ask_llm(f"Your code:\n{code_str}\nFailed tests:\n" + "\n".join(failures) +
                   "\nIn ONE sentence, write a lesson for your next attempt "
                   "(start with 'I failed because').").strip()

In [ ]:
def reflexion(max_attempts=4):
    memory = []
    for attempt in range(1, max_attempts + 1):
        code_str = write_code(memory)
        failures = evaluate(code_str)
        print(f"\n🧪 attempt {attempt}: {len(HIDDEN_TESTS) - len(failures)}/{len(HIDDEN_TESTS)} tests passed")
        if not failures:
            print("✅ all tests passed")
            return code_str, memory
        for f in failures:
            print("   ❌", f)
        lesson = reflect(code_str, failures)
        memory.append(lesson)
        print("   📝", lesson)
    return code_str, memory

In [ ]:
final_code, memory = reflexion()
print("\n📓 REFLECTION MEMORY:")
for m in memory:
    print(" -", m)

In [ ]:
print(final_code)

## สรุป

| องค์ประกอบ | ในโค้ด |
|---|---|
| Evaluator | `evaluate()` — hidden unit tests |
| Reflection | `reflect()` — บทเรียน 1 ประโยค "I failed because…" |
| Reflection memory | `memory` — สะสมทุกรอบ ใส่ใน prompt ผ่าน `write_code()` |
| Guard | `max_attempts` |

**ลองทำต่อ:** เรียก `write_code([])` ทุกรอบ (ไม่ใช้ memory) แล้วเทียบจำนวนรอบที่ใช้ — เห็นผลของ reflection memory ชัดเจน